In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
import pandas as pd
from datetime import datetime, timedelta

In [32]:
import boto3
from google.cloud import storage
from io import BytesIO

In [33]:
from pathlib import Path

credentials_path = Path('../config/lb-cloud-project-65885ec00848.json')

In [34]:
AWS_BUCKET_NAME = 'book-project-54641'
GCP_BUCKET_NAME = 'book-etl-project'
source = 'gcp'

In [6]:
today = datetime(2025, 5, 18)
TODAY_MINUS_1 = (today - timedelta(days=1)).strftime('%Y_%m_%d')

datetime.datetime(2025, 5, 18, 0, 0)

In [7]:
def extract_from_s3(bucket_name, file_name):
    print(f"Extracting {file_name} from AWS S3 bucket {bucket_name}")
    s3 = boto3.Session(profile_name='book-etl').client('s3')
    response = s3.get_object(Bucket=bucket_name, Key=file_name)
    data = response['Body'].read()
    return pd.read_csv(BytesIO(data))

def upload_to_s3(df, bucket_name, file_name):
    print(f"Uploading {file_name} to AWS S3 bucket {bucket_name}")
    # Convert DataFrame to CSV in memory
    buffer = BytesIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)
    # Upload using boto3 session
    s3 = boto3.Session(profile_name='book-etl').client('s3')
    s3.put_object(Bucket=bucket_name, Key=file_name, Body=buffer.getvalue())

    print("Upload to S3 completed.")

In [8]:
def extract_from_gcp(bucket_name, file_name):
    print(f"Extracting {file_name} from GCP bucket {bucket_name}")
    client = storage.Client.from_service_account_json(credentials_path)
    bucket = client.get_bucket(bucket_name)
    blob = bucket.blob(file_name)
    data = blob.download_as_bytes()
    return pd.read_csv(BytesIO(data))

def upload_to_gcp(df, bucket_name, file_name, credentials_path):
    print(f"Uploading {file_name} to GCP bucket {bucket_name}")
    # Convert DataFrame to CSV in memory
    buffer = BytesIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)
    # Upload using GCP storage client
    client = storage.Client.from_service_account_json(credentials_path)
    bucket = client.get_bucket(bucket_name)
    blob = bucket.blob(file_name)
    blob.upload_from_file(buffer, content_type='text/csv')

    print("Upload to GCP completed.")

### 1. Extract (E) data

In [9]:
# Calculate the three relevant dates: yesterday, day before yesterday, and two days ago
date_list = [(today - timedelta(days=i)).strftime('%Y_%m_%d') for i in range(1, 4)]
date_list

['2025_05_17', '2025_05_16', '2025_05_15']

In [10]:
file_names = [f"raw_transactions_{date}.csv" for date in date_list]
file_names

['raw_transactions_2025_05_17.csv',
 'raw_transactions_2025_05_16.csv',
 'raw_transactions_2025_05_15.csv']

In [11]:
df_ls = []
for file in file_names:
    if source == 'aws':
        df_ls.append(
            extract_from_s3(AWS_BUCKET_NAME, file)
        )
    elif source == 'gcp':
        df_ls.append(
            extract_from_gcp(GCP_BUCKET_NAME, file)
        )
# Concatenate the dataframes into one
df = pd.concat(df_ls, ignore_index=True)
df.shape

Extracting raw_transactions_2025_05_17.csv from GCP bucket book-etl-project
Extracting raw_transactions_2025_05_16.csv from GCP bucket book-etl-project
Extracting raw_transactions_2025_05_15.csv from GCP bucket book-etl-project


(60, 7)

In [12]:
df.head(20)

,transaction_id,user_id,amount,currency,date,status,payment_gateway
0,81,1081,60.00,USD,2025-05-17,completed,PayPal
1,82,1082,110.50,EUR,2025-05-17,completed,Stripe
2,83,1083,40.75,USD,2025-05-17,failed,Payoneer
3,84,1084,90.00,USD,2025-05-17,completed,Stripe
4,85,1085,47.00,EUR,2025-05-17,completed,PayPal
5,86,1086,25.50,USD,2025-05-17,completed,Stripe
6,87,1087,80.80,EUR,2025-05-17,completed,Payoneer
7,88,1088,100.00,USD,2025-05-17,completed,PayPal
8,89,1089,56.60,EUR,2025-05-17,completed,Stripe
9,90,1090,75.75,USD,2025-05-17,completed,Payoneer


In [35]:
df.dtypes

transaction_id       int64
user_id              int64
amount             float64
currency            object
date                object
status              object
payment_gateway     object
amount_usd         float64
dtype: object

### 2. Transform (T) data

In [13]:
# --- Step 1: Remove duplicates ---
df.drop_duplicates(subset=['user_id', 'amount', 'currency', 'date', 'status', 'payment_gateway'], inplace=True)
print(df.shape)

(60, 7)


In [14]:
# --- Step 2: Drop rows with missing values ---
df.dropna(subset=['amount'], inplace=True)
print(df.shape)

(57, 7)


In [15]:
# --- Step 3: Currency Conversion ---
conversion_rates = {
    'EUR': 1.1,  # EUR to USD
    'USD': 1.0,
}
df['amount_usd'] = df.apply(
    lambda x: round(x['amount'] * conversion_rates[x['currency']], 2), axis=1
)

In [16]:
df

,transaction_id,user_id,amount,currency,date,status,payment_gateway,amount_usd
0,81,1081,60.00,USD,2025-05-17,completed,PayPal,60.00
1,82,1082,110.50,EUR,2025-05-17,completed,Stripe,121.55
2,83,1083,40.75,USD,2025-05-17,failed,Payoneer,40.75
3,84,1084,90.00,USD,2025-05-17,completed,Stripe,90.00
4,85,1085,47.00,EUR,2025-05-17,completed,PayPal,51.70
5,86,1086,25.50,USD,2025-05-17,completed,Stripe,25.50
6,87,1087,80.80,EUR,2025-05-17,completed,Payoneer,88.88
7,88,1088,100.00,USD,2025-05-17,completed,PayPal,100.00
8,89,1089,56.60,EUR,2025-05-17,completed,Stripe,62.26
9,90,1090,75.75,USD,2025-05-17,completed,Payoneer,75.75


In [17]:
# Calculate success rate per date and payment_gateway
success_counts = df[df['status'] == 'completed'].groupby(['date', 'payment_gateway']).size()
total_counts = df.groupby(['date', 'payment_gateway']).size()

success_rate_df = pd.DataFrame({
    'date': total_counts.index.get_level_values('date'),
    'payment_gateway': total_counts.index.get_level_values('payment_gateway'),
    'total_transactions': total_counts.values,
    'successful_transactions': success_counts.reindex(total_counts.index, fill_value=0).values
})

success_rate_df['success_rate_percent'] = round(
    (success_rate_df['successful_transactions'] / success_rate_df['total_transactions']) * 100, 2
)

In [18]:
success_rate_df.sort_values(by=['date','payment_gateway'], ascending=True) #by='success_rate_percent', ascending=False)

,date,payment_gateway,total_transactions,successful_transactions,success_rate_percent
0,2025-05-15,PayPal,6,5,83.33
1,2025-05-15,Payoneer,6,6,100.00
2,2025-05-15,Stripe,7,6,85.71
3,2025-05-16,PayPal,6,6,100.00
4,2025-05-16,Payoneer,5,4,80.00
5,2025-05-16,Stripe,8,7,87.50
6,2025-05-17,PayPal,6,6,100.00
7,2025-05-17,Payoneer,6,4,66.67
8,2025-05-17,Stripe,7,7,100.00


In [19]:
# Calculate total amount per payment_gateway
total_amount = df.groupby(['date', 'payment_gateway'])['amount_usd'].sum()

# Calculate successful amount per payment_gateway
successful_amount = df[df['status'] == 'completed'].groupby(['date', 'payment_gateway'])['amount_usd'].sum()

# Combine into a DataFrame
amount_success_rate_df = pd.DataFrame({
    'date': total_amount.index.get_level_values('date'),
    'payment_gateway': total_amount.index.get_level_values('payment_gateway'),
    'total_amount_usd': total_amount.values,
    'successful_amount_usd': successful_amount.reindex(total_amount.index, fill_value=0).values
})

# Calculate success rate (percentage)
amount_success_rate_df['amount_success_rate_percent'] = round(
    (amount_success_rate_df['successful_amount_usd'] / amount_success_rate_df['total_amount_usd']) * 100, 2
)

In [20]:
amount_success_rate_df.sort_values(by=['date','payment_gateway'], ascending=True) #by='amount_success_rate_percent', ascending=False)

,date,payment_gateway,total_amount_usd,successful_amount_usd,amount_success_rate_percent
0,2025-05-15,PayPal,320.50,255.50,79.72
1,2025-05-15,Payoneer,415.09,415.09,100.00
2,2025-05-15,Stripe,506.17,461.17,91.11
3,2025-05-16,PayPal,360.69,360.69,100.00
4,2025-05-16,Payoneer,346.88,214.88,61.95
5,2025-05-16,Stripe,530.59,480.60,90.58
6,2025-05-17,PayPal,384.77,384.77,100.00
7,2025-05-17,Payoneer,399.88,254.63,63.68
8,2025-05-17,Stripe,572.71,572.71,100.00


### 3. Load (L) transformed data to bucket

In [29]:
folder_in_bucket = 'output'
if source == 'aws':
    file_name = f"{folder_in_bucket}/success_rate_df_{TODAY_MINUS_1}.csv"
    upload_to_s3(success_rate_df, AWS_BUCKET_NAME, file_name)
    print(f"Data successfully uploaded to {source} bucket: {file_name}")

    file_name = f"{folder_in_bucket}/amount_success_rate_df_{TODAY_MINUS_1}.csv"
    upload_to_s3(amount_success_rate_df, AWS_BUCKET_NAME, file_name)
    print(f"Data successfully uploaded to {source} bucket: {file_name}")
elif source == 'gcp':
    file_name = f"{folder_in_bucket}/success_rate_df_{TODAY_MINUS_1}.csv"
    upload_to_gcp(success_rate_df, GCP_BUCKET_NAME, file_name, credentials_path)
    print(f"Data successfully uploaded to {source} bucket: {file_name}")

    file_name = f"{folder_in_bucket}/amount_success_rate_df_{TODAY_MINUS_1}.csv"
    upload_to_gcp(amount_success_rate_df, GCP_BUCKET_NAME, file_name, credentials_path)
    print(f"Data successfully uploaded to {source} bucket: {file_name}")

Uploading output/success_rate_df_2025_05_17.csv to GCP bucket book-etl-project


Upload to GCP completed.
Data successfully uploaded to gcp bucket: output/success_rate_df_2025_05_17.csv
Uploading output/amount_success_rate_df_2025_05_17.csv to GCP bucket book-etl-project
Upload to GCP completed.
Data successfully uploaded to gcp bucket: output/amount_success_rate_df_2025_05_17.csv
